# PicoCal - In-time gated pooling: pushing 0.056 → toward the clean floor (notebook 17)

nb16 showed per-cell **timing** as a feature cuts σ_eff 0.0665 → 0.0564. The remaining gap to the clean-signal floor (~0.036, nb12) is leftover **pileup energy** that the model still pools over. Here we let the energy-weighted pooling **gate out out-of-time cells**: the pool weight becomes `energy × sigmoid(MLP(Δt, has_valid))`, so the network can suppress pileup cells in the energy sum — an explicit, learnable pile-up subtraction, exactly what PicoCal timing is for.

Ladder (all on the spacetime cache, kNN-25, 1–100 GeV, 3 seeds):
1. **+time tokens** — nb16 winner, EFN energy pooling (reproduces 0.0564)
2. **+ in-time gate** — gated energy pooling
3. **+ gate + Huber loss** — robustify the core the σ_eff calibration sees


In [1]:
import sys, copy, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import HistGradientBoostingRegressor

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import split, resolution, EPS

SEEDS = 3
cfg = {"d": 96, "nhead": 4, "layers": 3, "dropout": 0.1, "lr": 3e-4, "wd": 1e-4,
       "batch": 256, "epochs": 150, "patience": 25, "pair_hidden": 32, "huber_delta": 0.1}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
with open(repo / "data" / "cache" / "minbias_spacetime_knn25.pkl", "rb") as f:
    D = pickle.load(f)
y = D["y"]; Et = D["Etrue"]; agg = D["agg"]
keep = np.flatnonzero((Et >= 1.0) & (Et <= 100.0))
ktr, kva, kte = (keep[s] for s in split(len(keep)))
{"clusters": int(len(y)), "tok15_dim": int(D["tok15"][0].shape[1]), "device": DEVICE}

{'clusters': 89797, 'tok15_dim': 15, 'device': 'cuda'}

In [2]:
G = np.stack([agg[:, 0], agg[:, 3], np.log(agg[:, 2] + 1.0), agg[:, 1], agg[:, 4]], 1).astype(np.float32)
n_global = G.shape[1]
la, lb = np.polyfit(agg[ktr, 0], y[ktr], 1)
base_all = (la * agg[:, 0] + lb).astype(np.float32)
gb = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(agg[ktr], y[ktr])
BDT = float(resolution(np.exp(gb.predict(agg[kte])), Et[kte])["sigma_eff"])

N = len(y); in_dim = D["tok15"][0].shape[1]; maxL = max(t.shape[0] for t in D["tok15"])
X = np.zeros((N, maxL, in_dim), np.float32); M = np.zeros((N, maxL), np.bool_)
R = np.zeros((N, maxL, D["R"][0].shape[1]), np.float32); W = np.zeros((N, maxL), np.float32)
for i, t in enumerate(D["tok15"]):
    L = t.shape[0]; X[i, :L] = t; M[i, :L] = True; R[i, :L] = D["R"][i]
    e = np.expm1(np.clip(t[:, 0], 0, None)); W[i, :L] = e / (e.sum() + 1e-9)
cont = X[ktr][:, :, :9].reshape(-1, 9)[M[ktr].reshape(-1)]
mean = cont.mean(0); std = cont.std(0) + EPS
X[:, :, :9] = (X[:, :, :9] - mean) / std; X[~M] = 0.0
gmean = G[ktr].mean(0); gstd = G[ktr].std(0) + EPS
Gn = ((G - gmean) / gstd).astype(np.float32)

T = lambda z: torch.from_numpy(z).to(DEVICE)
Xt, Mt, Rt, Wt, Gt = T(X), T(M), T(R), T(W), T(Gn)
Bt = T(base_all).unsqueeze(1); Yt = T(y.astype(np.float32)).unsqueeze(1)
{"BDT": round(BDT, 4), "n_train": int(len(ktr)), "n_test": int(len(kte)), "maxL": int(maxL)}

{'BDT': 0.1253, 'n_train': 57056, 'n_test': 12227, 'maxL': 25}

In [3]:
def pair_feats(R):
    rx, ry, le = R[..., 0], R[..., 1], R[..., 2]
    dx = rx.unsqueeze(2) - rx.unsqueeze(1); dy = ry.unsqueeze(2) - ry.unsqueeze(1)
    dR = torch.sqrt(dx * dx + dy * dy + 1e-6)
    esum = le.unsqueeze(2) + le.unsqueeze(1); emin = torch.minimum(le.unsqueeze(2), le.unsqueeze(1))
    return torch.stack([dx, dy, dR, esum, emin], -1)

class PairEmbed(nn.Module):
    def __init__(self, nhead, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(5, hidden), nn.GELU(), nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, nhead))
    def forward(self, pf): return self.net(pf).permute(0, 3, 1, 2).contiguous()

class PMHA(nn.Module):
    def __init__(self, d, nh, drop):
        super().__init__(); self.h = nh; self.dh = d // nh
        self.q = nn.Linear(d, d); self.k = nn.Linear(d, d); self.v = nn.Linear(d, d); self.o = nn.Linear(d, d); self.drop = nn.Dropout(drop)
    def forward(self, x, U, kv):
        B, L, d = x.shape
        q = self.q(x).view(B, L, self.h, self.dh).transpose(1, 2); k = self.k(x).view(B, L, self.h, self.dh).transpose(1, 2); v = self.v(x).view(B, L, self.h, self.dh).transpose(1, 2)
        s = (q @ k.transpose(-2, -1)) / (self.dh ** 0.5) + U
        s = s.masked_fill((~kv).view(B, 1, 1, L), -1e9)
        return self.o((self.drop(s.softmax(-1)) @ v).transpose(1, 2).reshape(B, L, d))

class Block(nn.Module):
    def __init__(self, d, nh, drop):
        super().__init__(); self.n1 = nn.LayerNorm(d); self.attn = PMHA(d, nh, drop); self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d)); self.drop = nn.Dropout(drop)
    def forward(self, x, U, kv):
        x = x + self.drop(self.attn(self.n1(x), U, kv)); return x + self.drop(self.ff(self.n2(x)))

class PairT(nn.Module):
    def __init__(self, use_gate):
        super().__init__(); d = cfg["d"]; self.use_gate = use_gate
        self.embed = nn.Linear(in_dim, d); self.pair = PairEmbed(cfg["nhead"], cfg["pair_hidden"])
        self.blocks = nn.ModuleList([Block(d, cfg["nhead"], cfg["dropout"]) for _ in range(cfg["layers"])])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + n_global, d), nn.ReLU(), nn.Dropout(cfg["dropout"]), nn.Linear(d, 1))
        if use_gate:
            self.gate = nn.Sequential(nn.Linear(2, 16), nn.GELU(), nn.Linear(16, 1))
    def forward(self, x, m, w, g, base, R):
        U = self.pair(pair_feats(R)); h = self.embed(x)
        for blk in self.blocks: h = blk(h, U, m)
        pw = w
        if self.use_gate:
            gate = torch.sigmoid(self.gate(R[..., 3:5]).squeeze(-1)) * m.float()
            pw = w * gate
            pw = pw / pw.sum(1, keepdim=True).clamp(min=1e-9)
        p = self.norm((h * pw.unsqueeze(-1)).sum(1))
        return base + self.head(torch.cat([p, g], 1))

In [4]:
def train_eval(use_gate, loss_kind, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = PairT(use_gate).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    def lossf(p, t):
        return nn.functional.huber_loss(p, t, delta=cfg["huber_delta"]) if loss_kind == "huber" else nn.functional.mse_loss(p, t)
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs):
            b = torch.from_numpy(idx[j:j + bs]).to(DEVICE)
            yield Xt[b], Mt[b], Wt[b], Gt[b], Bt[b], Rt[b], Yt[b]
    def run(idx):
        out = []
        with torch.no_grad():
            for X_, m, w, g, base, R_, _ in batches(idx, 512, False):
                out.append(model(X_, m, w, g, base, R_).cpu().numpy().ravel())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; c = 0
        with torch.no_grad():
            for X_, m, w, g, base, R_, yb in batches(kva, 512, False):
                s += lossf(model(X_, m, w, g, base, R_), yb).item(); c += 1
        return s / max(c, 1)
    best = 1e9; bstate = None; wait = 0
    for ep in range(cfg["epochs"]):
        model.train()
        for X_, m, w, g, base, R_, yb in batches(ktr, cfg["batch"], True):
            opt.zero_grad(); lossf(model(X_, m, w, g, base, R_), yb).backward(); opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= cfg["patience"]: break
    model.load_state_dict(bstate); model.eval()
    a, b = np.polyfit(run(kva), y[kva], 1)
    pe = np.exp(a * run(kte) + b)
    return float(resolution(pe, Et[kte])["sigma_eff"]), pe

LADDER = [("+time tokens (nb16 winner)", False, "mse"),
          ("+ in-time gate", True, "mse"),
          ("+ gate + Huber", True, "huber")]
res17 = {}; best_pe = {}; rows = []
t0 = time.time()
for name, ug, lk in LADDER:
    vals = []; pe_last = None
    for s in range(SEEDS):
        sig, pe = train_eval(ug, lk, s); vals.append(sig); pe_last = pe
        print(f"  {name} seed {s}: {sig:.4f}", flush=True)
    best_pe[name] = pe_last; mean, sd = float(np.mean(vals)), float(np.std(vals)); res17[name] = vals
    rows.append({"config": name, "sigma_eff": round(mean, 4), "std": round(sd, 4), "BDT": round(BDT, 4)})
    print(f"{name}: {mean:.4f} +/- {sd:.4f}", flush=True)
print(f"elapsed {time.time()-t0:.0f}s")
summary17 = pd.DataFrame(rows)
summary17

  +time tokens (nb16 winner) seed 0: 0.0565


  +time tokens (nb16 winner) seed 1: 0.0555


  +time tokens (nb16 winner) seed 2: 0.0572


+time tokens (nb16 winner): 0.0564 +/- 0.0007


  + in-time gate seed 0: 0.0592


...

  + gate + Huber seed 1: 0.0490


  + gate + Huber seed 2: 0.0502


+ gate + Huber: 0.0500 +/- 0.0007


elapsed 7870s


,config,sigma_eff,std,BDT
0,+time tokens (nb16 winner),0.0564,0.0007,0.1253
1,+ in-time gate,0.0578,0.0011,0.1253
2,+ gate + Huber,0.0500,0.0007,0.1253


In [5]:
import plotly.graph_objects as go
d = summary17
fig = go.Figure(go.Bar(x=d["config"], y=d["sigma_eff"], error_y=dict(type="data", array=d["std"]),
                       marker_color=["#8c8c8c", "#1f77b4", "#2ca02c"],
                       text=[f"{v:.4f}" for v in d["sigma_eff"]], textposition="outside"))
fig.add_hline(y=BDT, line_dash="dash", line_color="crimson", annotation_text=f"BDT {BDT:.4f}")
fig.add_hline(y=0.04, line_dash="dot", line_color="green", annotation_text="target 0.04")
fig.add_hline(y=0.036, line_dash="dot", line_color="gray", annotation_text="clean floor ~0.036")
fig.update_layout(template="plotly_white", height=460, yaxis_title="sigma_eff (min-bias)",
                  title="In-time gated pooling vs the timing baseline")
fig.show()

In [6]:
def sig_bins(pe, tr, n=8):
    edges = np.quantile(tr, np.linspace(0, 1, n + 1)); cx, sy = [], []
    for i in range(n):
        hi = edges[i + 1] + (1e-6 if i == n - 1 else 0.0); mk = (tr >= edges[i]) & (tr < hi)
        if mk.sum() >= 20: cx.append(float(np.median(tr[mk]))); sy.append(float(resolution(pe[mk], tr[mk])["sigma_eff"]))
    return np.array(cx), np.array(sy)
best = summary17.sort_values("sigma_eff").iloc[0]["config"]
fig = go.Figure()
for name, col in [("+time tokens (nb16 winner)", "#8c8c8c"), (best, "#2ca02c")]:
    cx, sy = sig_bins(best_pe[name], Et[kte]); fig.add_trace(go.Scatter(x=cx, y=sy, mode="lines+markers", name=name, line=dict(color=col)))
fig.add_hline(y=0.04, line_dash="dot", line_color="green", annotation_text="0.04")
fig.update_layout(template="plotly_white", height=430, xaxis_title="E_true [GeV] (bin median)", yaxis_title="sigma_eff",
                  title="Resolution vs energy: timing baseline vs best gated model")
fig.show()